# 4) EE-2.2 Active, reactive and complex power — Exercises

**Goals:** complex power with the conjugate, power factor and its sign convention, load currents from P and power factor, power factor correction, power flows and losses on a line, the angle needed to transfer power (exact vs. DC approximation), and the P–δ / Q–V coupling behind fast power flow methods.

Conventions: RMS phasors; S = V·I\*; a load **consuming** reactive power has Q > 0 (inductive, **lagging** power factor). Powers in W/var/VA unless stated.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import cmath
import math

import numpy as np
import pandas as pd

F = 50.0
OMEGA = 2 * math.pi * F

### Warm-ups

**1. Complex power**

Write `complex_power(V, I)` = V·I\*. Check it against the power absorbed by an impedance, |I|²·Z, for an inductive and a capacitive load.

In [2]:
def complex_power(V: complex, I: complex) -> complex:
    """Complex power S = V * conj(I) (VA)."""
    ### BEGIN SOLUTION
    return V * np.conj(I)
    ### END SOLUTION

In [3]:
for Z in [10 + 15j, 10 - 15j]:
    I = 230 / Z
    assert np.isclose(complex_power(230, I), abs(I) ** 2 * Z), f"S must equal |I|^2 Z for Z = {Z}"
assert complex_power(230, 230 / (10 + 15j)).imag > 0, "an inductive load consumes reactive power"
assert complex_power(230, 230 / (10 - 15j)).imag < 0, "a capacitive load produces it"

**2. Power factor and its sign**

Write `power_factor(S)` returning `(pf, kind)` with pf = P/|S| and kind `"lagging"` (Q > 0, inductive), `"leading"` (Q < 0, capacitive) or `"unity"`.

In [4]:
def power_factor(S: complex) -> tuple[float, str]:
    """Power factor P/|S| and 'lagging', 'leading' or 'unity'."""
    ### BEGIN SOLUTION
    pf = S.real / abs(S)
    if np.isclose(S.imag, 0, atol=1e-9 * max(1.0, abs(S))):
        return float(pf), "unity"
    return float(pf), "lagging" if S.imag > 0 else "leading"
    ### END SOLUTION

In [5]:
pf, kind = power_factor(8000 + 6000j)
assert math.isclose(pf, 0.8) and kind == "lagging"
assert power_factor(8000 - 6000j)[1] == "leading"
assert power_factor(5000 + 0j) == (1.0, "unity")

### Core

**3. Load current from P and power factor**

Equipment is specified by active power and power factor, not by impedance. Write `load_current(P, pf, V, lagging=True)` returning the current phasor (A). Hint: build S from P and pf, then use S = V·I\*.

In [6]:
def load_current(P: float, pf: float, V: complex, lagging: bool = True) -> complex:
    """Current phasor (A) drawn by a load of active power P (W) at power factor pf."""
    ### BEGIN SOLUTION
    Q = P * math.tan(math.acos(pf)) * (1 if lagging else -1)
    return np.conj((P + 1j * Q) / V)
    ### END SOLUTION

In [7]:
I = load_current(10_000, 0.8, 230)
S = complex_power(230, I)
assert np.isclose(S, 10_000 + 7_500j), "10 kW at pf 0.8 lagging: |S| = 12.5 kVA, Q = 7.5 kvar"
assert math.isclose(abs(I), 12_500 / 230)
assert abs(load_current(10_000, 0.6, 230)) > abs(I), "lower power factor, more current for the same power"
print(f"pf 0.8: {abs(I):.1f} A | pf 0.6: {abs(load_current(10_000, 0.6, 230)):.1f} A")

pf 0.8: 54.3 A | pf 0.6: 72.5 A


**4. Power factor correction**

Write `capacitor_for_pf(P, Q, V_rms, pf_target)`: the parallel capacitance (F) that raises a lagging load's power factor to `pf_target`. Then check the result by computing the corrected power factor, and verify that the active power is unchanged.

In [8]:
def capacitor_for_pf(P: float, Q: float, V_rms: float, pf_target: float, f: float = F) -> float:
    """Parallel capacitance (F) that corrects a load (P, Q) to pf_target (lagging or unity)."""
    ### BEGIN SOLUTION
    Q_c = Q - P * math.tan(math.acos(pf_target))
    return Q_c / (V_rms ** 2 * 2 * math.pi * f)
    ### END SOLUTION

In [9]:
P_load, Q_load = 10_000, 7_500
for target in [0.9, 0.95, 1.0]:
    C = capacitor_for_pf(P_load, Q_load, 230, target)
    S_cap = -1j * 230 ** 2 * OMEGA * C                 # a capacitor produces reactive power
    S_total = P_load + 1j * Q_load + S_cap
    assert math.isclose(power_factor(S_total)[0], target), f"corrected pf should be {target}"
    assert math.isclose(S_total.real, P_load), "active power unchanged"
    print(f"target {target:.2f}: C = {C * 1e6:6.1f} uF, source current {abs(S_total) / 230:5.1f} A")
assert capacitor_for_pf(P_load, Q_load, 230, 0.95) < capacitor_for_pf(P_load, Q_load, 230, 1.0)

target 0.90: C =  159.9 uF, source current  48.3 A
target 0.95: C =  253.5 uF, source current  45.8 A
target 1.00: C =  451.3 uF, source current  43.5 A


**5. Flows and losses on a line**

Write `line_flows(V1, V2, Z)` returning a dict with `S12` (sent from bus 1 into the line), `S21` (sent from bus 2 into the line) and `losses` (S12 + S21). Check that the losses equal |I|²·Z, and that a purely reactive line has **no active losses** but does consume reactive power.

In [10]:
def line_flows(V1: complex, V2: complex, Z: complex) -> dict:
    """Complex power entering the line at each end, and the line's losses (VA)."""
    ### BEGIN SOLUTION
    I12 = (V1 - V2) / Z
    S12 = V1 * np.conj(I12)
    S21 = V2 * np.conj(-I12)
    return {"S12": S12, "S21": S21, "losses": S12 + S21}
    ### END SOLUTION

In [11]:
V1, V2 = 230e3 * cmath.exp(1j * math.radians(8)), 225e3 + 0j
flows = line_flows(V1, V2, 4 + 40j)
I12 = (V1 - V2) / (4 + 40j)
assert np.isclose(flows["losses"], abs(I12) ** 2 * (4 + 40j)), "losses = |I|^2 Z"
assert flows["S12"].real > 0 and flows["S21"].real < 0, "active power flows from the leading angle"

lossless = line_flows(V1, V2, 40j)
assert np.isclose(lossless["losses"].real, 0, atol=1e-3), "no resistance, no active losses"
assert lossless["losses"].imag > 0, "the line consumes reactive power"
print({k: f"{v / 1e6:.2f} MVA" for k, v in flows.items()})

{'S12': '182.37+23.10j MVA', 'S21': '-179.81+2.45j MVA', 'losses': '2.56+25.55j MVA'}


**6. Interpretation**

Generators and transformers are rated in **MVA**, not MW, and many utilities charge industrial customers for reactive energy. Use this lesson's results to explain both.

<!-- BEGIN SOLUTION -->
The limit of a generator winding, transformer or cable is set by **current** (heating, I²R) and by **voltage** (insulation) — together, by the apparent power |S| = |V||I|, regardless of how that current splits into active and reactive parts. A 500 MVA generator can deliver 500 MW at unity power factor, but only 400 MW if it must also supply 300 Mvar; the rating has to be stated in MVA. Reactive power drawn by a customer makes the utility carry extra current through every line and transformer upstream: it occupies capacity that could carry active power, raises I²R losses (tasks 3–4: current grows as 1/pf, losses as 1/pf²) and causes voltage drops. Charging for reactive energy (or requiring a minimum power factor) passes that cost to the customer and gives an incentive to correct locally with capacitors — far cheaper than reinforcing the network.
<!-- END SOLUTION -->

### Challenge

**7. What angle does a transfer need?**

For a lossless line with reactance X, write `angle_for_transfer(P, V1, V2, X)` returning the exact angle (degrees) from P = V₁V₂ sin δ / X, and `dc_angle(P, V1, V2, X)` from the DC approximation P = V₁V₂ δ / X. If P exceeds the maximum transferable power, `angle_for_transfer` must raise `ValueError`. Compare both for a 220 kV line (X = 40 Ω), and check what a parallel line (X halved) does.

In [12]:
def angle_for_transfer(P: float, V1: float, V2: float, X: float) -> float:
    """Exact angle (degrees) needed to send P (W) over a lossless line."""
    ### BEGIN SOLUTION
    s = P * X / (V1 * V2)
    if s > 1:
        raise ValueError(f"P = {P / 1e6:.0f} MW exceeds the maximum {V1 * V2 / X / 1e6:.0f} MW")
    return math.degrees(math.asin(s))
    ### END SOLUTION


def dc_angle(P: float, V1: float, V2: float, X: float) -> float:
    """Angle (degrees) from the DC approximation P = V1 V2 delta / X."""
    ### BEGIN SOLUTION
    return math.degrees(P * X / (V1 * V2))
    ### END SOLUTION

In [13]:
V, X = 220e3, 40.0
rows = []
for P_MW in [100, 400, 800, 1100]:
    exact, approx = angle_for_transfer(P_MW * 1e6, V, V, X), dc_angle(P_MW * 1e6, V, V, X)
    rows.append({"P_MW": P_MW, "exact_deg": exact, "dc_deg": approx, "dc_error_pct": 100 * (approx - exact) / exact})
angles = pd.DataFrame(rows)
assert (angles.dc_deg <= angles.exact_deg).all(), "the DC approximation underestimates the angle"
assert angles.dc_error_pct.is_monotonic_decreasing, "the error grows (more negative) with the transfer"
try:
    angle_for_transfer(1300e6, V, V, X)
    raise AssertionError("1,300 MW exceeds the 1,210 MW limit and must raise")
except ValueError:
    pass
assert math.isclose(dc_angle(400e6, V, V, X / 2), dc_angle(400e6, V, V, X) / 2), "parallel line: half the angle"
angles.round(2)

   P_MW  exact_deg  dc_deg  dc_error_pct
0   100       4.74    4.74         -0.11
1   400      19.30   18.94         -1.88
2   800      41.39   37.88         -8.47
3  1100      65.38   52.09        -20.33

**8. Angles carry P, magnitudes carry Q**

At δ = 10° on the same line, raise |V₁| by 5 % while keeping the angle fixed. Write `sensitivity_to_V1(V, X, delta_deg, rise)` returning the **relative** change in P₁₂ and in Q₁₂ (using `line_flows`).

In [14]:
def sensitivity_to_V1(V: float, X: float, delta_deg: float, rise: float) -> dict:
    """Relative change of P12 and Q12 when |V1| rises by `rise` (e.g. 0.05) at a fixed angle."""
    ### BEGIN SOLUTION
    V2 = V + 0j
    base = line_flows(V * cmath.exp(1j * math.radians(delta_deg)), V2, 1j * X)["S12"]
    high = line_flows(V * (1 + rise) * cmath.exp(1j * math.radians(delta_deg)), V2, 1j * X)["S12"]
    return {"P_change": high.real / base.real - 1, "Q_change": high.imag / base.imag - 1}
    ### END SOLUTION

In [15]:
sens = sensitivity_to_V1(220e3, 40.0, 10, 0.05)
assert math.isclose(sens["P_change"], 0.05), "P scales with |V1|: +5 %"
assert sens["Q_change"] > 2.0, "Q reacts far more strongly to the voltage magnitude"
print(f"+5 % |V1| at 10 degrees: P {100 * sens['P_change']:+.1f} %, Q {100 * sens['Q_change']:+.0f} %")

+5 % |V1| at 10 degrees: P +5.0 %, Q +351 %


**9. Interpretation**

Summarise tasks 7 and 8 for a grid operator: which variable would you change to move more active power over a line, and which to raise a sagging voltage? How do these two facts justify (a) the DC power flow used in market models and (b) the fast "decoupled" power flow methods of Module 04?

<!-- BEGIN SOLUTION -->
**Active power follows angles; reactive power follows voltage magnitudes.** Raising |V₁| by 5 % changes P by only 5 % but more than quadruples Q₁₂ here (+350 %), while transferring more active power needs a larger angle difference (task 7) and barely changes magnitudes on a reactive line (EE-2.1). So an operator moves active power by changing generation (which changes angles) or network impedance (a parallel line halves the needed angle), and supports voltage with reactive power — generator excitation, capacitors, reactors, inverters. (a) The **DC power flow** keeps only the P–δ part: magnitudes fixed at nominal, losses and Q ignored, sin δ ≈ δ. Task 7 shows the cost: errors well under 1 % at the small angles of normal operation, growing to several percent at 30° and far more near the 90° limit — accurate enough for markets and planning, but it says nothing about voltages. (b) **Decoupled power flow** methods use the same insight inside Newton's method: they solve P against angles and Q against magnitudes as two smaller, almost independent systems with a constant Jacobian, which is much faster for transmission grids with high X/R ratios.
<!-- END SOLUTION -->